# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [319]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [320]:
df  = pd.read_csv("data/AviationData.csv",encoding="cp1252",low_memory=False)

In [321]:
# get the data structure 
df.shape

(88889, 31)

In [322]:
# inspect the nans values

missing = df.isnull().sum()
print(missing)

# getdatatypes
dtype = df.dtypes
print(dtype)

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [323]:
# loading dataframe
df.head(5)

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

Aircrafts that are 40 years and above 
 - extract the year from the event date
 - calculate the sum of data below 1983
 - drop those values

In [324]:
# create a column with year 
df["year"] = df["Event.Date"].str[:4].astype(int)

In [325]:
df["year"]

0        1948
1        1962
2        1974
3        1977
4        1979
         ... 
88884    2022
88885    2022
88886    2022
88887    2022
88888    2022
Name: year, Length: 88889, dtype: int64

In [326]:
# airplanes below 40 years
below_th = (df['year'] < 1983).sum()
print(below_th)

3600


In [327]:
# drop all rows below the 40 years
df = df.drop(df[df['year'] < 1983].index)
below_th = (df['year'] < 1983).sum()
print(below_th)

0


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

Creating total passengers column
 - first get the null valuables in all the related tables
 - fill them with relevant values
 - calculate the estimated total passengers in the plane
 - drop unrealistic rows eg with 0 passengers
 - create a row with sum of serious/fatal injuries
 - create a row with rate of injuries from the total passangers


In [328]:
# calculate null values 
missing_fatal = df['Total.Fatal.Injuries'].isnull().sum()
print(missing_fatal)
missing_serious = df['Total.Serious.Injuries'].isnull().sum()
print(missing_serious)
missing_minor = df['Total.Minor.Injuries'].isnull().sum()
print(missing_minor)
tt_uninjured = df['Total.Uninjured'].isnull().sum()
print(tt_uninjured)



11376
12481
11905
5903


In [329]:
# fill with values 
df['Total.Fatal.Injuries']=df['Total.Fatal.Injuries'].fillna(0)
df['Total.Serious.Injuries']=df['Total.Serious.Injuries'].fillna(0)
df['Total.Minor.Injuries']=df['Total.Minor.Injuries'].fillna(0)
df['Total.Uninjured']=df['Total.Uninjured'].fillna(0)


In [330]:
# create total passenges including the crew estimated  pasengers row
df['total_passengers'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + df['Total.Minor.Injuries'] + df['Total.Uninjured']


In [331]:
# find values that are below or equal to 0 in total passengers and drop them 
print((df['total_passengers'] <= 0).sum())

1309


In [332]:
# drop irrevant values to our study
df = df.drop(df[df['total_passengers'] == 0].index)

In [333]:
# create total injuries row 
df['total_injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + df['Total.Minor.Injuries']


In [334]:
# create total rate row  to see the injury rate in the plane
df['injury_rate'] = df['total_injuries'] / df['total_passengers']

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

AirCraft damage cleaning procedure
- find the shape of the column
- find the null values
- fill them appropriately
- create column tracking if it was destroyed or not (1, 0)

In [335]:
# find shape
df['Aircraft.damage'].value_counts()

Aircraft.damage
Substantial    61506
Destroyed      17479
Minor           2369
Unknown           92
Name: count, dtype: int64

In [336]:
#find null values
print(df['Aircraft.damage'].isnull().sum())

2534


In [337]:
# fill null values
df['Aircraft.damage']=df['Aircraft.damage'].fillna('Unknown')

In [338]:
df["Aircraft_destroyed"] = (df["Aircraft.damage"] == "Destroyed").astype(int)
df["Aircraft_destroyed"].value_counts()

Aircraft_destroyed
0    66501
1    17479
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

cleaning make column
- first get the value count
- find the null values
- convert the data to title case
- remove makes with less that 50 

In [339]:
# value count
df["Make"].value_counts()

Make
Cessna                20874
Piper                 11288
CESSNA                 4838
Beech                  4057
PIPER                  2813
                      ...  
Raum                      1
Reitz                     1
Levitsky                  1
Pitenpol Aircamper        1
Kolb Twin Star            1
Name: count, Length: 8091, dtype: int64

In [340]:
# find null values
# print(df["Make"].isnull().sum())
df["Make"] = df["Make"].fillna('unKnown')

In [341]:
# convert the data to title case
df["Make"] = df["Make"].str.strip().str.title()

In [342]:
# find makes with lower than 50
make_counts = df["Make"].value_counts()
rare_makes = make_counts[make_counts < 50]
rare_makes_count = (make_counts < 50).sum()
print(rare_makes_count)

7352


In [343]:
df = df[df["Make"].map(make_counts) >= 50]
df

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,year,total_passengers,total_injuries,injury_rate,Aircraft_destroyed
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,...,3.0,VMC,Approach,Probable Cause,NaN,1983,4.0,1.0,0.25,0
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,...,2.0,VMC,Landing,Probable Cause,NaN,1983,2.0,0.0,0.00,0
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,...,1.0,VMC,Takeoff,Probable Cause,NaN,1983,1.0,0.0,0.00,0
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,...,0.0,VMC,Approach,Probable Cause,NaN,1983,2.0,2.0,1.00,0
3605,20001214X41951,Accident,CHI83LA074,1983-01-01,"SPRINGBROOK, WI",United States,NaN,NaN,NaN,SPRINGBROOK,...,2.0,VMC,Landing,Probable Cause,NaN,1983,2.0,0.0,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88882,20221222106486,Accident,CEN23LA068,2022-12-21,"Reserve, LA",United States,NaN,NaN,NaN,NaN,...,1.0,NaN,NaN,NaN,27-12-2022,2022,2.0,1.0,0.50,0
88883,20221228106502,Accident,GAA23WA046,2022-12-22,"Brasnorte,",Brazil,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,28-12-2022,2022,1.0,1.0,1.00,0
88884,20221227106491,Accident,ERA23LA093,2022-12-26,"Annapolis, MD",United States,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,29-12-2022,2022,1.0,1.0,1.00,0
88886,20221227106497,Accident,WPR23LA075,2022-12-26,"Payson, AZ",United States,341525N,1112021W,PAN,PAYSON,...,1.0,VMC,NaN,NaN,27-12-2022,2022,1.0,0.0,0.00,0


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

cleaning the models column
- calculate value counts
- convert the titles to title case
- look for null values
- remove null values
  

In [344]:
df['Model'].value_counts()

Model
152          2226
172          1641
172N         1093
PA-28-140     868
172M          759
             ... 
180R            1
G-102CS         1
M4-220          1
VEGA-1          1
SGU-2-22K       1
Name: count, Length: 6166, dtype: int64

In [345]:
df = df.dropna(subset=["Model"])

In [346]:
print(df['Model'].isnull().sum())

0


In [347]:
df = df[df["Model"].notna()].copy()
df["Model"] = df["Model"].str.strip().str.title()

In [348]:
# common makes
df.groupby("Model")["Make"].nunique().sort_values(ascending=False).head(20)

Model
500       7
G-164A    7
G-164B    6
Aa-5      5
G-164     5
G-164C    5
Aa-5B     5
690A      4
S2R       4
Aa1B      4
700       4
269C      4
690C      4
600       4
690B      4
100       4
112       4
300       4
G164-B    4
Aa-5A     4
Name: Make, dtype: int64

In [349]:
df["Plane_Type_ID"] = (
    df["Make"].str.strip().str.title() + "_" +
    df["Model"].str.strip().str.upper()
)
df['Plane_Type_ID'].value_counts()


Plane_Type_ID
Cessna_152          2226
Cessna_172          1641
Cessna_172N         1093
Piper_PA-28-140      868
Cessna_172M          759
                    ... 
Airbus_A320-271N       1
Bell_H13G              1
Boeing_737-8           1
Bell_UH-1D             1
Airbus_A319-132        1
Name: count, Length: 6706, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [313]:
df["Engine.Type"] = df["Engine.Type"].str.strip().str.title()
df["Weather.Condition"] = df["Weather.Condition"].str.strip().str.title()
df["Purpose.of.flight"] = df["Purpose.of.flight"].str.strip().str.title()
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].str.strip().str.title()
# df["Number.of.Engines"] = df["Number.of.Engines"].str.replace(',', '', regex=False).astype(float)

In [350]:
# df[df["Number.of.Engines"] == 0][["Make", "Model", "Aircraft.Category"]].head()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [351]:
df.isnull().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     40
Country                     192
Latitude                  44070
Longitude                 44075
Airport.Code              30196
Airport.Name              28338
Injury.Severity               0
Aircraft.damage               0
Aircraft.Category         49182
Registration.Number        1015
Make                          0
Model                         0
Amateur.Built                55
Number.of.Engines          4170
Engine.Type                4727
FAR.Description           49305
Schedule                  59840
Purpose.of.flight          4441
Air.carrier               57337
Total.Fatal.Injuries          0
Total.Serious.Injuries        0
Total.Minor.Injuries          0
Total.Uninjured               0
Weather.Condition          2850
Broad.phase.of.flight     19722
Report.Status              4341
Publication.Date          12001
year    

In [352]:
df = df.drop(df.columns[[6,7,8,9,12,13,18,20,22,27,28,29,30]], axis = 1)

In [353]:
df

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Injury.Severity,Aircraft.damage,Make,Model,...,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,year,total_passengers,total_injuries,injury_rate,Aircraft_destroyed,Plane_Type_ID
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,Non-Fatal,Substantial,Cessna,182P,...,0.0,0.0,1.0,3.0,1983,4.0,1.0,0.25,0,Cessna_182P
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,Non-Fatal,Substantial,Cessna,182Rg,...,0.0,0.0,0.0,2.0,1983,2.0,0.0,0.00,0,Cessna_182RG
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,Non-Fatal,Substantial,Cessna,182P,...,0.0,0.0,0.0,1.0,1983,1.0,0.0,0.00,0,Cessna_182P
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,Non-Fatal,Substantial,Piper,Pa-28R-200,...,0.0,0.0,2.0,0.0,1983,2.0,2.0,1.00,0,Piper_PA-28R-200
3605,20001214X41951,Accident,CHI83LA074,1983-01-01,"SPRINGBROOK, WI",United States,Non-Fatal,Substantial,Cessna,140,...,0.0,0.0,0.0,2.0,1983,2.0,0.0,0.00,0,Cessna_140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88882,20221222106486,Accident,CEN23LA068,2022-12-21,"Reserve, LA",United States,Minor,Unknown,Grumman American Avn. Corp.,Aa-5B,...,0.0,1.0,0.0,1.0,2022,2.0,1.0,0.50,0,Grumman American Avn. Corp._AA-5B
88883,20221228106502,Accident,GAA23WA046,2022-12-22,"Brasnorte,",Brazil,Fatal,Unknown,Air Tractor,At502,...,1.0,0.0,0.0,0.0,2022,1.0,1.0,1.00,0,Air Tractor_AT502
88884,20221227106491,Accident,ERA23LA093,2022-12-26,"Annapolis, MD",United States,Minor,Unknown,Piper,Pa-28-151,...,0.0,1.0,0.0,0.0,2022,1.0,1.0,1.00,0,Piper_PA-28-151
88886,20221227106497,Accident,WPR23LA075,2022-12-26,"Payson, AZ",United States,Non-Fatal,Substantial,American Champion Aircraft,8Gcbc,...,0.0,0.0,0.0,1.0,2022,1.0,0.0,0.00,0,American Champion Aircraft_8GCBC


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [354]:
df.to_csv("data/aviation_cleaned.csv", index=False)